# 01 · English STT benchmark (Colab GPU)

Runs every shortlisted speech-to-text model on the same clips and appends results to
`results.csv` in your Google Drive (`MyDrive/local-voice-research/results`), so a disconnect never loses finished work.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU` (or any GPU offered).
2. Key icon in the left sidebar → add secret **`HF_TOKEN`** with your Hugging Face token → switch on *Notebook access*.

**How to run:** run cells top to bottom. Each model section is independent.
If a section errors, screenshot the error and skip to the next section (30-minute rule).

## 1 · Setup (run this again after any *Restart session*)

In [1]:
import os, sys, subprocess, importlib
from google.colab import drive, userdata

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

ROOT = "/content/drive/MyDrive/local-voice-research"
DATA, RESULTS = f"{ROOT}/data", f"{ROOT}/results"
os.makedirs(DATA, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)

if os.path.isdir("/content/lvr"):
    subprocess.run(["git", "-C", "/content/lvr", "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/Kiranrossi/local-voice-research", "/content/lvr"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "whisper_normalizer"])

sys.path.insert(0, "/content/lvr/src")
import harness as H; importlib.reload(H)
import gc, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: set Runtime > Change runtime type > T4 GPU")
print("Text normaliser:", H.NORMALISER_NAME)

def run_all(model, name, sets=("librispeech", "svarah", "own_en", "noise"), **kw):
    """Runs one model on every test set that has been prepared. Returns {test_set: results row}."""
    rows = {}
    for ts in sets:
        m = f"{DATA}/{ts}/manifest.jsonl"
        if os.path.exists(m):
            rows[ts] = H.evaluate(model, name, m, ts, RESULTS, **kw)
        else:
            print(f"skip {ts}: not prepared yet")
    return rows

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

GPU: Tesla T4
Text normaliser: whisper_english


## 2 · Prepare test sets (once, about 5 minutes)

- **librispeech**: 100 random clips from LibriSpeech test-clean (sanity check against published numbers)
- **svarah**: 150 random clips of Indian-accented English (AI4Bharat Svarah, gated)
- **noise**: 10 clips with no speech (silence and noise), to measure hallucination

All clips are 1 to 30 s, saved as 16 kHz mono WAV in your Drive. Random sampling uses seed 42, so it is reproducible.

In [6]:
import io, json
import numpy as np, soundfile as sf, librosa
from datasets import load_dataset, Audio

def save_clips(ds, name, n, min_s=1.0, max_s=30.0):
    out = f"{DATA}/{name}"
    if os.path.exists(f"{out}/manifest.jsonl"):
        print(f"{name}: already prepared"); return
    os.makedirs(out, exist_ok=True)
    audio_col = [k for k, f in ds.features.items() if isinstance(f, Audio)][0]
    text_col = next(k for k in ("text", "transcription", "sentence", "normalized_text") if k in ds.column_names)
    meta_cols = [k for k, f in ds.features.items()
                 if k not in (audio_col, text_col) and getattr(f, "dtype", None) == "string"]
    ds = ds.cast_column(audio_col, Audio(decode=False)).shuffle(seed=42)
    rows = []
    for ex in ds:
        a = ex[audio_col]
        raw = a["bytes"] if a.get("bytes") else open(a["path"], "rb").read()
        audio, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
        audio = audio.mean(axis=1)
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        dur = len(audio) / 16000
        if not (min_s <= dur <= max_s) or not ex[text_col]:
            continue
        cid = f"{name}_{len(rows):03d}"
        path = f"{out}/{cid}.wav"
        sf.write(path, audio, 16000, subtype="PCM_16")
        rows.append({"id": cid, "audio_filepath": path, "text": ex[text_col], "duration": round(dur, 3),
                     **{k: ex.get(k) for k in meta_cols}})
        if len(rows) >= n:
            break
    H.write_manifest(rows, f"{out}/manifest.jsonl")
    print(f"{name}: {len(rows)} clips, {sum(r['duration'] for r in rows)/60:.1f} min of audio")

# LibriSpeech test-clean (try the Open ASR Leaderboard copy first, then the original)
try:
    libri = load_dataset("hf-audio/esb-datasets-test-only-sorted", "librispeech", split="test.clean")
except Exception as e:
    print("ESB copy failed, trying openslr:", type(e).__name__)
    libri = load_dataset("openslr/librispeech_asr", "clean", split="test")
save_clips(libri, "librispeech", 100)

# Svarah (Indian-accented English)
svarah = load_dataset("ai4bharat/Svarah", split="test")
save_clips(svarah, "svarah", 150)

# No-speech clips for the hallucination test
if not os.path.exists(f"{DATA}/noise/manifest.jsonl"):
    H.make_noise_set(f"{DATA}/noise")
print("noise: 10 clips")

librispeech: already prepared
svarah: already prepared
noise: 10 clips


### Your own recordings

Upload the extracted `recordings` folder from the recorder ZIP to
`MyDrive/local-voice-research/data/recordings`, then run this cell.
It splits the clips into **own_en** (the 10 English sentences, used now) and **own_ta** (the 30 Tamlish sentences, used on Day 2).

In [7]:
src = f"{DATA}/recordings/manifest.jsonl"
if not os.path.exists(src):
    print("Not found:", src, "\nUpload the 'recordings' folder to MyDrive/local-voice-research/data/ first.")
else:
    clips = H.read_manifest(src)  # makes paths absolute
    for name, prefix in (("own_en", "en_"), ("own_ta", "ta_")):
        part = [c for c in clips if c["id"].startswith(prefix)]
        H.write_manifest(part, f"{DATA}/{name}/manifest.jsonl")
        print(f"{name}: {len(part)} clips, {sum(c['duration'] for c in part):.0f} s of audio")
    missing = [c["id"] for c in clips if not os.path.exists(c["audio_filepath"])]
    print("Missing audio files:", missing or "none")

own_en: 10 clips, 74 s of audio
own_ta: 30 clips, 144 s of audio
Missing audio files: none


In [8]:
# Quick look at the Svarah sample (who is speaking)
import pandas as pd
sv = pd.read_json(f"{DATA}/svarah/manifest.jsonl", lines=True)
print(sv.columns.tolist())
for col in ("gender", "age-group", "primary_language", "native_language"):
    if col in sv:
        print(f"\n{col}:\n", sv[col].value_counts().head(10).to_string())

['id', 'audio_filepath', 'text', 'duration', 'gender', 'age-group', 'primary_language', 'native_place_state', 'native_place_district', 'highest_qualification', 'job_category', 'occupation_domain']

gender:
 gender
Female    78
Male      72

age-group:
 age-group
30-45    53
18-30    51
45-60    32
60+      14

primary_language:
 primary_language
Maithili     13
Kannada      11
Urdu         11
Tamil        11
Nepali       10
Konkani      10
Hindi        10
Bodo          9
Telugu        9
Malayalam     8


## 3 · Whisper large-v3-turbo (expected LibriSpeech WER about 2 to 3%)

If our LibriSpeech WER is far from that, the setup is wrong. Stop and send me the output.

In [9]:
from transformers import pipeline

def hf_pipe(model_id, **kw):
    try:
        return pipeline("automatic-speech-recognition", model=model_id, dtype=torch.float16, device="cuda:0", **kw)
    except TypeError:  # older transformers
        return pipeline("automatic-speech-recognition", model=model_id, torch_dtype=torch.float16, device="cuda:0", **kw)

class HFWhisper:
    def __init__(self, model_id): self.model_id = model_id
    def load(self):
        if not hasattr(self, "pipe"): self.pipe = hf_pipe(self.model_id)
    def transcribe(self, clip):
        out = self.pipe({"raw": clip["audio"].copy(), "sampling_rate": 16000},
                        generate_kwargs={"language": "en", "task": "transcribe"})
        return out["text"]

model = HFWhisper("openai/whisper-large-v3-turbo")
rows = run_all(model, "whisper-large-v3-turbo", variant="809M", runtime="transformers")
if "svarah" in rows:
    print("\nHardest Svarah clips for Whisper:")
    H.worst_examples(RESULTS, rows["svarah"]["run_id"], n=3)
del model; free_gpu()

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

  25/100 clips
  50/100 clips
  75/100 clips
  100/100 clips
whisper-large-v3-turbo | librispeech: WER 1.01% (95% CI 0.53%-1.62%) | RTF 0.0772 | median latency 0.5075 s | peak 1.84 GB
  25/150 clips
  50/150 clips
  75/150 clips
  100/150 clips
  125/150 clips
  150/150 clips
whisper-large-v3-turbo | svarah: WER 7.48% (95% CI 5.60%-9.95%) | RTF 0.0618 | median latency 0.3641 s | peak 1.84 GB
whisper-large-v3-turbo | own_en: WER 12.50% (95% CI 6.25%-18.35%) | RTF 0.052 | median latency 0.3729 s | peak 1.84 GB
whisper-large-v3-turbo | noise: hallucinated on 8/10 no-speech clips | RTF 0.045 | peak 1.84 GB

Hardest Svarah clips for Whisper:
[svarah_116] errors=14
  REF: are maghe sagrati khepa sung losar sa ke wa barahimi jung sakadawa chaite dussai
  HYP: i am going to ask you a question

[svarah_139] errors=7
  REF: can you delete diabetics dezire sugar free juice and fiama gel bathing bar from my shopping cart in addition to this please remove 250 gm alpino classic peanut butter crunch


## 4 · Moonshine Streaming (medium 245M and tiny 34M)

In [12]:
class HFMoonshine:
    def __init__(self, model_id): self.model_id = model_id

    def load(self):
        if not hasattr(self, "pipe"): self.pipe = hf_pipe(self.model_id)

    def transcribe(self, clip):
        audio = clip["audio"]
        pad = (-len(audio)) % 320  # the streaming frontend needs a whole number of frames; pads at most 20 ms of silence
        audio = np.pad(audio, (0, pad))
        max_tokens = int(clip["duration"] * 6.5) + 10
        return self.pipe({"raw": audio, "sampling_rate": 16000},
                         generate_kwargs={"max_new_tokens": max_tokens})["text"]

for size, params in (("medium", "245M"), ("tiny", "34M")):
    model = HFMoonshine(f"moonshine-ai/moonshine-streaming-{size}")
    run_all(model, f"moonshine-streaming-{size}", variant=params, runtime="transformers",
            notes="audio zero-padded to a multiple of 320 samples")
    del model; free_gpu()

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

  25/100 clips
  50/100 clips
  75/100 clips
  100/100 clips
moonshine-streaming-medium | librispeech: WER 1.44% (95% CI 0.84%-2.16%) | RTF 0.0929 | median latency 0.6776 s | peak 1.35 GB
  25/150 clips
  50/150 clips
  75/150 clips
  100/150 clips
  125/150 clips
  150/150 clips
moonshine-streaming-medium | svarah: WER 11.94% (95% CI 8.55%-16.64%) | RTF 0.0846 | median latency 0.4919 s | peak 1.36 GB
moonshine-streaming-medium | own_en: WER 15.38% (95% CI 10.34%-21.74%) | RTF 0.0843 | median latency 0.6173 s | peak 1.19 GB
moonshine-streaming-medium | noise: hallucinated on 0/10 no-speech clips | RTF 0.1936 | peak 1.16 GB


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  176MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer.json:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

  25/100 clips
  50/100 clips
  75/100 clips
  100/100 clips
moonshine-streaming-tiny | librispeech: WER 3.80% (95% CI 2.72%-5.15%) | RTF 0.0483 | median latency 0.3539 s | peak 0.71 GB
  25/150 clips
  50/150 clips
  75/150 clips
  100/150 clips
  125/150 clips
  150/150 clips
moonshine-streaming-tiny | svarah: WER 23.57% (95% CI 16.72%-34.30%) | RTF 0.0446 | median latency 0.2425 s | peak 0.71 GB
moonshine-streaming-tiny | own_en: WER 39.42% (95% CI 26.26%-57.43%) | RTF 0.046 | median latency 0.3239 s | peak 0.66 GB
moonshine-streaming-tiny | noise: hallucinated on 0/10 no-speech clips | RTF 0.0329 | peak 0.66 GB


## 5 · Qwen3-ASR (0.6B and 1.7B)

Installs the `qwen-asr` package. If the install or the model errors, screenshot it and move on to section 6.

In [13]:
!pip install -q -U qwen-asr

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from qwen_asr import Qwen3ASRModel

class QwenASR:
    def __init__(self, model_id): self.model_id = model_id
    def load(self):
        if not hasattr(self, "m"):
            self.m = Qwen3ASRModel.from_pretrained(self.model_id, dtype=torch.float16, device_map="cuda:0",
                                                   max_new_tokens=256)
    def transcribe(self, clip):
        return self.m.transcribe(audio=clip["path"], language="English")[0].text

for size in ("0.6B", "1.7B"):
    model = QwenASR(f"Qwen/Qwen3-ASR-{size}")
    run_all(model, f"qwen3-asr-{size}", variant=size, runtime="qwen-asr")
    del model; free_gpu()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


preprocessor_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

...

qwen3-asr-1.7B | svarah: WER 8.92% (95% CI 6.89%-11.50%) | RTF 0.1458 | median latency 0.8612 s | peak 4.32 GB


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


qwen3-asr-1.7B | own_en: WER 24.04% (95% CI 14.29%-36.67%) | RTF 0.1329 | median latency 0.9688 s | peak 4.2 GB


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


qwen3-asr-1.7B | noise: hallucinated on 10/10 no-speech clips | RTF 0.0408 | peak 4.17 GB


In [4]:
import json, glob
print("Output files:", sorted(os.path.basename(p) for p in glob.glob(f"{RESULTS}/outputs/*own_en*")), "\n")
for pat in ["*qwen3-asr-1-7B_own_en*", "*whisper-large-v3-turbo_own_en*", "*qwen3-asr-1-7B_noise*"]:
    files = sorted(glob.glob(f"{RESULTS}/outputs/{pat}"))
    if not files:
        print("No file matches", pat); continue
    print("=====", os.path.basename(files[-1]))
    for line in open(files[-1], encoding="utf-8"):
        r = json.loads(line)
        print(f"[{r['id']}] REF: {r['ref_norm']}\n         HYP: {r['hyp_norm']}")

Output files: ['20261029-091605_whisper-large-v3-turbo_own_en.jsonl', '20261029-092526_moonshine-streaming-medium_own_en.jsonl', '20261029-092712_moonshine-streaming-tiny_own_en.jsonl', '20261029-093843_qwen3-asr-0-6B_own_en.jsonl', '20261029-094511_qwen3-asr-1-7B_own_en.jsonl'] 

===== 20261029-094511_qwen3-asr-1-7B_own_en.jsonl
[en_01] REF: transfer 2499 to karthikeyan on 15th august
         HYP: transfer rupees 200499 to kartikayn on 15th august
[en_02] REF: book a train from chennai egmore to tiruchirappalli for 3rd december
         HYP: book a train from chennai egmore to tiruchrapalli for 3rd december
[en_03] REF: call 98401 23456 and ask for mister subramanian
         HYP: call 9840123456 and ask for mister subramanian
[en_04] REF: my complaint reference number is tn 4471 b check the status
         HYP: my complaint reference number is t n 4471 b check the status
[en_05] REF: set a reminder for 7 45 pm to pay the lic premium of 12350
         HYP: set a reminder for 745 p m 

In [6]:
import subprocess, importlib, sys
print(subprocess.run(["git", "-C", "/content/lvr", "pull"], capture_output=True, text=True).stdout)
print(subprocess.run(["git", "-C", "/content/lvr", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
import harness as H; importlib.reload(H)
print("Normaliser:", H.NORMALISER_NAME, "| has rescore_all:", hasattr(H, "rescore_all"))

Updating c8cfcc2..777c715
Fast-forward
 src/harness.py | 87 +++++++++++++++++++++++++++++++++++++++++++++++++++++++---
 1 file changed, 83 insertions(+), 4 deletions(-)

777c715 Scoring v2: Indian-English formatting rules (currency, digit grouping, spelled letters) and rescore_all

Normaliser: whisper_english+indian_v2 | has rescore_all: True


In [7]:
import pandas as pd
rows = H.rescore_all(RESULTS)
df = pd.DataFrame(rows)
cols = ["model", "test_set", "wer", "wer_ci_low", "wer_ci_high", "hallucination_rate", "rtf", "latency_median_s", "peak_mem_gb"]
print(df[cols].to_string(index=False))

Re-scored 20 runs with whisper_english+indian_v2 -> /content/drive/MyDrive/local-voice-research/results/results_scored.csv
                     model    test_set     wer wer_ci_low wer_ci_high hallucination_rate    rtf latency_median_s peak_mem_gb
moonshine-streaming-medium librispeech  0.0144     0.0084      0.0216                    0.0929           0.6776        1.35
  moonshine-streaming-tiny librispeech   0.038     0.0272      0.0515                    0.0483           0.3539        0.71
            qwen3-asr-0.6B librispeech  0.0149     0.0096       0.021                    0.1739           1.2671        1.88
            qwen3-asr-1.7B librispeech  0.0139     0.0084        0.02                    0.1725           1.3089        4.71
    whisper-large-v3-turbo librispeech  0.0101     0.0053      0.0162                    0.0772           0.5075        1.84
moonshine-streaming-medium       noise                                               0.0 0.1936           1.0425        1.16
  

## 6 · NVIDIA Parakeet TDT 0.6B (v2 English, v3 multilingual)

1. Run the install cell (about 5 minutes).
2. **Runtime → Restart session**.
3. Run the **Setup** cell (section 1) again, then the cell below. Data and results are on Drive, so nothing is lost.

In [8]:
!pip install -q "nemo_toolkit[asr]"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [9]:
import nemo.collections.asr as nemo_asr

class Parakeet:
    def __init__(self, model_id): self.model_id = model_id
    def load(self):
        if not hasattr(self, "m"):
            self.m = nemo_asr.models.ASRModel.from_pretrained(self.model_id).eval()
    def transcribe(self, clip):
        try:
            out = self.m.transcribe([clip["path"]], batch_size=1, verbose=False)
        except TypeError:
            out = self.m.transcribe([clip["path"]], batch_size=1)
        if isinstance(out, tuple): out = out[0]
        h = out[0]
        return h.text if hasattr(h, "text") else str(h)

for v in ("v2", "v3"):
    model = Parakeet(f"nvidia/parakeet-tdt-0.6b-{v}")
    run_all(model, f"parakeet-tdt-0.6b-{v}", variant="600M", quantization="fp32", runtime="nemo")
    del model; free_gpu()

parakeet-tdt-0.6b-v2.nemo:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

[NeMo I 2026-10-29 09:54:58 mixins:194] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-10-29 09:54:59 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-10-29 09:54:59 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    use_

[NeMo I 2026-10-29 09:55:03 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-10-29 09:55:03 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-10-29 09:55:03 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-10-29 09:55:14 save_restore_connector:287] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v2/snapshots/ae9ad07059c7c739ffaf932226a8fe64ae2620b0/parakeet-tdt-0.6b-v2.nemo.


[NeMo W 2026-10-29 09:55:15 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-29 09:55:15 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-10-29 09:55:17 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-29 09:55:17 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is

...

parakeet-tdt-0.6b-v3 | svarah: WER 9.13% (95% CI 7.10%-11.42%) | RTF 0.0198 | median latency 0.117 s | peak 2.79 GB


[NeMo W 2026-10-29 10:00:26 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-29 10:00:26 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-10-29 10:00:26 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-29 10:00:26 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is

parakeet-tdt-0.6b-v3 | own_en: WER 14.42% (95% CI 6.80%-22.83%) | RTF 0.0165 | median latency 0.1209 s | peak 2.68 GB


[NeMo W 2026-10-29 10:00:28 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-29 10:00:28 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-10-29 10:00:28 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-29 10:00:28 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is

parakeet-tdt-0.6b-v3 | noise: hallucinated on 0/10 no-speech clips | RTF 0.0205 | peak 2.67 GB


## 7 · IBM Granite Speech 5.0 470M TurboCTC (optional, experimental)

This model is very new and its loading code is not verified. If it errors, skip it: it will be evaluated on published evidence only.

In [11]:
import torch
from transformers import pipeline

def hf_pipe(model_id, **kw):
    try:
        return pipeline("automatic-speech-recognition", model=model_id, dtype=torch.float16, device="cuda:0", **kw)
    except TypeError:
        return pipeline("automatic-speech-recognition", model=model_id, torch_dtype=torch.float16, device="cuda:0", **kw)

class HFGeneric:
    def __init__(self, model_id): self.model_id = model_id
    def load(self):
        if not hasattr(self, "pipe"): self.pipe = hf_pipe(self.model_id, trust_remote_code=True)
    def transcribe(self, clip):
        return self.pipe({"raw": clip["audio"].copy(), "sampling_rate": 16000})["text"]

model = HFGeneric("ibm-granite/granite-speech-5.0-470m-turboctc")
run_all(model, "granite-speech-5.0-470m-turboctc", variant="470M", runtime="transformers")
del model; free_gpu()

config.json:   0%|          | 0.00/921 [00:00<?, ?B/s]

ValueError: The checkpoint you are trying to load has model type `granite_speech5_ctc` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [12]:
rows = H.rescore_all(RESULTS)
print(pd.DataFrame(rows)[["model","test_set","wer","hallucination_rate","rtf","latency_median_s","peak_mem_gb"]].to_string(index=False))
!cd /content/drive/MyDrive/local-voice-research && zip -qr /content/stt_results.zip results && echo "zipped"
from google.colab import files; files.download("/content/stt_results.zip")

Re-scored 28 runs with whisper_english+indian_v2 -> /content/drive/MyDrive/local-voice-research/results/results_scored.csv
                     model    test_set     wer hallucination_rate    rtf latency_median_s peak_mem_gb
moonshine-streaming-medium librispeech  0.0144                    0.0929           0.6776        1.35
  moonshine-streaming-tiny librispeech   0.038                    0.0483           0.3539        0.71
      parakeet-tdt-0.6b-v2 librispeech  0.0101                    0.0224           0.1223         5.0
      parakeet-tdt-0.6b-v3 librispeech  0.0159                    0.0172           0.1244         5.1
            qwen3-asr-0.6B librispeech  0.0149                    0.1739           1.2671        1.88
            qwen3-asr-1.7B librispeech  0.0139                    0.1725           1.3089        4.71
    whisper-large-v3-turbo librispeech  0.0101                    0.0772           0.5075        1.84
moonshine-streaming-medium       noise                       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8 · Results so far

In [13]:
import pandas as pd
df = pd.read_csv(f"{RESULTS}/results.csv")
df = df[df.task == "stt"]
cols = ["model", "test_set", "wer", "wer_ci_low", "wer_ci_high", "hallucination_rate",
        "rtf", "latency_median_s", "peak_mem_gb", "device_name"]
pd.set_option("display.width", 200)
print(df[cols].to_string(index=False))
print(f"\nResults file: {RESULTS}/results.csv  (upload it to the chat)")

                     model    test_set    wer  wer_ci_low  wer_ci_high  hallucination_rate    rtf  latency_median_s  peak_mem_gb device_name
    whisper-large-v3-turbo librispeech 0.0101      0.0053       0.0162                 NaN 0.0538            0.4040         1.84    Tesla T4
    whisper-large-v3-turbo      svarah 0.0748      0.0560       0.0995                 NaN 0.0570            0.3452         1.84    Tesla T4
    whisper-large-v3-turbo       noise    NaN         NaN          NaN                 0.8 0.0472            0.2473         1.84    Tesla T4
    whisper-large-v3-turbo librispeech 0.0101      0.0053       0.0162                 NaN 0.0772            0.5075         1.84    Tesla T4
    whisper-large-v3-turbo      svarah 0.0748      0.0560       0.0995                 NaN 0.0618            0.3641         1.84    Tesla T4
    whisper-large-v3-turbo      own_en 0.1250      0.0625       0.1835                 NaN 0.0520            0.3729         1.84    Tesla T4
    whisper-l